##### On commence par importer les bibliothèques nécessaires. Attention : NE PAS EFFACER !

In [1]:
%run Import_bibliotheques.ipynb
%run Arduino.ipynb

In [2]:
# Connexion à l'Arduino
connexion_arduino()

# Durée d'acquisition : des boutons de durées prédéfinies (fiables à 100 % :
# uniquement des clics, aucune saisie clavier) + un champ libre pour une durée
# personnalisée. Champ vide = dernière durée utilisée.
if 'duree_acquisition' not in globals():
    duree_acquisition = float('inf')
champ_duree = widgets.Text(value='', description='Durée (s) :',
                           placeholder='vide = dernière durée',
                           layout=widgets.Layout(width='24em'))


def applique_duree():
    """Relit le champ durée ; s'il est vide, garde la durée précédente."""
    global duree_acquisition
    saisie = champ_duree.value.strip()
    if saisie:
        duree_acquisition = float(saisie.replace(',', '.'))    # virgule tolérée


def duree_preset(d):
    """Bouton de durée : mémorise la durée et l'affiche dans le champ."""
    def _clic(_b):
        global duree_acquisition
        duree_acquisition = d
        champ_duree.value = '' if d == float('inf') else str(d).replace('.', ',')
    return _clic


# Durées courantes proposées en un clic, puis « continu » (∞).
boutons_duree = []
for d in [0.1, 0.5, 1, 2, 5, 10]:
    b = widgets.Button(description=f"{str(d).replace('.', ',')} s",
                       layout=widgets.Layout(width='5em'))
    b.on_click(duree_preset(d))
    boutons_duree.append(b)
b_infini = widgets.Button(description='continu ∞', layout=widgets.Layout(width='8em'))
b_infini.on_click(duree_preset(float('inf')))
boutons_duree.append(b_infini)

# Graphique plotly : créé et affiché immédiatement (une seule fois), puis
# réutilisé à chaque clic. NB : l'affichage doit se faire ici, au niveau de la
# cellule — affiché depuis le rappel du bouton, il atterrirait dans une zone de
# sortie détachée invisible de JupyterLab.
clear_output(wait=True)
graphique = initialise_graphique()
# Échelle en abscisse accordée à la durée totale AVANT l'affichage : le graphe
# vide montre d'emblée l'horizon de la mesure (0 → durée) au lieu d'une échelle
# d'attente. En durée infinie, échelle automatique.
regle_abscisse(duree_acquisition)
display(graphique)

# Bouton Démarrer / Arrêter. L'acquisition tourne dans un thread de fond : le
# kernel étant mono-tâche, sans cela le clic « Arrêter » ne serait jamais
# traité avant la fin de l'enregistrement.
thread_acquisition = None
start_button = widgets.Button(description='Démarrer')
# Statut de l'acquisition : les print émis depuis le thread de fond ne
# parviennent pas à la cellule, on affiche le dernier message dans un Label.
statut = widgets.Label(value='')


def on_button_click(b):
    global thread_acquisition
    if thread_acquisition is not None and thread_acquisition.is_alive():
        b.description = 'Démarrer'
        arreter_acquisition()
    else:
        applique_duree()    # durée saisie dans le champ (défaut : dernière utilisée)
        # Échelle posée ICI, sur le thread principal, AVANT le lancement de
        # l'acquisition : les mises à jour de layout depuis le thread de fond
        # sont appliquées de manière aléatoire par le frontend plotly.
        regle_abscisse(duree_acquisition)
        b.description = 'Arrêter'
        statut.value = 'Enregistrement en cours…'

        def tache():
            global thread_acquisition
            try:
                statut.value = 'Enregistrement en cours…'
                trace_temps_reel(duree_acquisition)
            finally:
                statut.value = message_acquisition
                b.description = 'Démarrer'
                thread_acquisition = None

        thread_acquisition = threading.Thread(target=tache, daemon=True)
        thread_acquisition.start()


start_button.on_click(on_button_click)
# Affichage unique, durée d'abord : durées pré-réglées, puis champ, bouton, statut
display(widgets.HBox(boutons_duree))
display(widgets.HBox([champ_duree, start_button, statut]))
# Focus sur le champ UNE FOIS le widget affiché (il n'existe pas avant)
champ_duree.focus()    # la saisie d'une durée personnalisée commence aussitôt



FigureWidget({
    'data': [{'hovertemplate': 't = %{x:.3f} s<br>U = %{y:.3f} V<extra></extra>',
              'mode': 'lines',
              'name': 'Tension',
              'type': 'scatter',
              'uid': '4abcd083-ad5a-4a8b-bfd9-2e3eba09b054',
              'x': [],
              'y': []}],
    'layout': {'height': 420,
               'hovermode': 'x unified',
               'margin': {'b': 35, 'l': 45, 'r': 20, 't': 40},
               'template': '...',
               'title': {'text': 'Tension en fonction du temps'},
               'width': 900,
               'xaxis': {'autorange': True, 'gridcolor': '#dddddd', 'showgrid': True, 'title': {'text': 'Temps (s)'}},
               'yaxis': {'gridcolor': '#dddddd', 'range': [0, 5], 'showgrid': True, 'title': {'text': 'Tension (V)'}}}
})